# Actividad — SVM con kernel RBF para reconocimiento de dígitos
**Aprendizaje Supervisado · Sesión 10 · Máquinas de vectores de soporte para clasificación (SVM)**

**Qué haremos.** En la presentación vimos que SVM busca la frontera con el **margen más ancho**, que
el parámetro $C$ controla el compromiso entre margen y errores, y que el **truco del kernel** (RBF en
particular, con su parámetro $\gamma$) permite fronteras curvas sin construir variables nuevas a mano.
Hoy aplicas todo eso al reconocimiento de **dígitos manuscritos** (imágenes de $8\times8$ píxeles, vía
`sklearn.datasets.load_digits`), un problema de **10 clases**.

Trabaja **de forma individual**. Las celdas con `...` están incompletas: reemplaza los puntos
suspensivos con el código que falta (el comentario `✏️` te da una pista de qué se espera).


## Parte 0 — Los datos

Cada imagen es una matriz de $8\times 8$ píxeles en escala de grises, aplanada a un vector de 64
variables (una por píxel, con valores de 0 a 16). Hay 10 clases (dígitos 0 a 9) y 1 797 imágenes en
total.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

digitos = load_digits()
X, y = digitos.data, digitos.target
print("X:", X.shape, "  clases:", np.unique(y))
pd.Series(y).value_counts().sort_index()


In [ ]:
fig, axs = plt.subplots(2, 10, figsize=(12, 3))
for clase in range(10):
    idx_clase = np.where(y == clase)[0]
    for fila in range(2):
        ax = axs[fila, clase]
        ax.imshow(digitos.images[idx_clase[fila]], cmap="gray_r")
        ax.axis("off")
        if fila == 0:
            ax.set_title(str(clase))
plt.suptitle("Dos ejemplos de cada dígito"); plt.tight_layout(); plt.show()


✏️ **Pregunta 0.1.** (a) ¿Cuántas imágenes hay de cada dígito (aproximadamente)? ¿El conjunto está
balanceado entre las 10 clases? (b) Mirando los ejemplos de la celda anterior, ¿qué par de dígitos te
parece, a simple vista, más fácil de confundir entre sí?

**Respuesta:**


## Parte 1 — SVM con kernel lineal  *(presentación A1–A4)*

Dividimos en entrenamiento/prueba de forma estratificada y ajustamos un `Pipeline` con `StandardScaler`
$+$ `SVC(kernel="linear")`.


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

svm_lineal = ...     # ✏️ make_pipeline(StandardScaler(), SVC(kernel="linear")).fit(Xtr, ytr)
print("Exactitud SVM lineal (prueba):", round(svm_lineal.score(Xte, yte), 4))


✏️ **Pregunta 1.1.** ¿Qué exactitud obtuvo el SVM lineal en el conjunto de prueba? Dado que cada
imagen tiene 64 variables (píxeles) y solo 10 clases, ¿te sorprende que un modelo **lineal** (sin
kernel curvo) ya funcione razonablemente bien?

**Respuesta:**


## Parte 2 — SVM con kernel RBF  *(presentación C1–C2)*

Repetimos con `SVC(kernel="rbf")` (valores por defecto de $C$ y $\gamma$) y comparamos.


In [ ]:
svm_rbf = ...     # ✏️ make_pipeline(StandardScaler(), SVC(kernel="rbf")).fit(Xtr, ytr)
print("Exactitud SVM lineal:", round(svm_lineal.score(Xte, yte), 4))
print("Exactitud SVM RBF (por defecto):", round(svm_rbf.score(Xte, yte), 4))


✏️ **Pregunta 2.1.** ¿Mejora el resultado al usar el kernel RBF (con sus valores por defecto) en vez
del lineal? ¿La mejora te parece grande o pequeña, dado lo que ya respondiste en la Pregunta 1.1?

**Respuesta:**


## Parte 3 — Ajuste de $C$ y $\gamma$ con `GridSearchCV`  *(presentación B3–B4, C5–C6)*

Buscamos la mejor combinación de $C$ y $\gamma$ sobre una rejilla, con validación cruzada de 5
pliegues, y visualizamos la exactitud de validación como un mapa de calor.


In [ ]:
malla = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.005, 0.01, 0.05, 0.1]}
pipe_rbf = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
busqueda = ...     # ✏️ GridSearchCV(pipe_rbf, malla, cv=kf, scoring="accuracy", n_jobs=-1).fit(Xtr, ytr)

print("Mejor combinación:", busqueda.best_params_, "   exactitud de validación:", round(busqueda.best_score_, 4))
print("Exactitud en prueba:", round(busqueda.score(Xte, yte), 4))

resultados = pd.DataFrame(busqueda.cv_results_)
tabla = resultados.pivot_table(index="param_svc__C", columns="param_svc__gamma", values="mean_test_score")

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(tabla.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(tabla.columns))); ax.set_xticklabels(tabla.columns)
ax.set_yticks(range(len(tabla.index))); ax.set_yticklabels(tabla.index)
ax.set_xlabel("gamma"); ax.set_ylabel("C"); ax.set_title("Exactitud de validación cruzada")
fig.colorbar(im, ax=ax, label="exactitud")
for i in range(len(tabla.index)):
    for j in range(len(tabla.columns)):
        ax.text(j, i, f"{tabla.values[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8)
plt.show()


✏️ **Pregunta 3.1.** (a) ¿Qué combinación de $C$ y $\gamma$ eligió `GridSearchCV`, y qué exactitud de
validación alcanzó? (b) Mirando el mapa de calor, ¿la zona de buenos resultados es amplia (muchas
combinaciones de $C$/$\gamma$ dan una exactitud parecida) o estrecha (solo una combinación destaca
claramente)? (c) ¿Qué le pasa a la exactitud cuando $\gamma$ es muy grande (0.1) y $C$ también es
grande? Relaciónalo con lo que viste en la presentación sobre el sobreajuste de $\gamma$ grande.

**Respuesta:**


## Parte 4 — Evaluación final y confusiones más comunes  *(presentación D1–D2)*

Evaluamos el modelo final (el elegido por `GridSearchCV`) **una sola vez** sobre el conjunto de
prueba, con su matriz de confusión completa (10×10).


In [ ]:
modelo_final = busqueda.best_estimator_
pred = modelo_final.predict(Xte)
print(classification_report(yte, pred))

fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(yte, pred, cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Matriz de confusión — modelo final (prueba)")
plt.show()


In [ ]:
cm = confusion_matrix(yte, pred)
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(np.argmax(cm), cm.shape)
print(f"Confusión más frecuente: dígito real {i} clasificado como {j}  ({cm[i, j]} caso(s))")

errores = np.argwhere(cm > 0)
print("Todas las confusiones observadas (real -> predicho : número de casos):")
for fi, fj in errores:
    print(f"  {fi} -> {fj} : {cm[fi, fj]}")


✏️ **Pregunta 4.1.** (a) ¿Cuántos errores comete en total el modelo final sobre las 450 imágenes de
prueba? (b) Lista dos o tres pares de dígitos que se confunden en la matriz. (c) Para uno de esos
pares, da una explicación razonable de por qué se parecen visualmente (puedes volver a la Parte 0 para
comparar las imágenes).

**Respuesta:**
